This it the collab notebook to implement a RAG.

The inputs or source/knowledge base are the pdfs in the folder /RAG/documents.

The pdfs can be text or images.

The vector db is Chroma db under /RAG/db.

The hosting is using gradio.

The RAG pipeline is separated into a data ingestion, that populates the db.

The embeddings that creates the embeddings.

The retrieval that retrieves the data from db.

The  generation that generates the output.

# Create requirements. txt file


In [28]:
packages = [#'numpy',
            'pypdf',
            'fastapi',
            'uvicorn',
            'gradio',
            'groq',
            'langchain',
            #'hugging_face_hub',
            'langchain-community',
            'langchain-huggingface',
            'sentence_transformers',
            'chromadb',
            'python-dotenv',

            ]

with open("requirements.txt", "w") as f:
  for package in packages:
    f.write(f"{package}\n")

In [29]:
!pip install --upgrade -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.8 MB/s eta 0:00:00


## Load the Enviornment files that have the keys



In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
#
groq_key = os.getenv("GROQ_KEY")
hugging_face_key = os.getenv("HUGGING_FACE_KEY")

if groq_key.startswith("gsk_"):
  print(f"Groq Key loaded!")
else:
  print(f"Failed to load Groq Key!")

if hugging_face_key.startswith("hf_"):
  print(f"Hugging Key loaded!")
else:
  print(f"Failed to load Hugging Face Key!")

Groq Key loaded!
Hugging Key loaded!


## The Data ingestion

In [4]:
from langchain_community.document_loaders import PyPDFLoader
import os
def load_documents(knowledge_base_path:str):
  documents = []
  for file in os.listdir(knowledge_base_path):
    file_path = os.path.join(knowledge_base_path, file)
    loader = PyPDFLoader(file_path)
    doc = loader.load()
    documents.extend(doc)
    print(f"Loaded {len(doc)} pages from {file_path}")

  print(f"Loaded total {len(documents)} documents")
  return documents

## Create Chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunkizer(documents:list, chunk_size:int, chunk_overlap:int):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )
  chunks = text_splitter.split_documents(documents)
  print(f"Chunizer: num documents: {len(documents)}, Chunk Sz: {chunk_size} Chunk OverLap: {chunk_overlap}")
  print(f"Chunkizer created {len(chunks)} chunks")
  return chunks

## Storing the emeddings in vector DB

In [9]:
import shutil
import os
from langchain_community.vectorstores import Chroma
import chromadb # Added this import

def add_to_vector_db(vector_db_path:str, chunks:list, embedding_model, purge: bool):
  # Create a persistent client for the specific path. This ensures each vector store
  # operates with its own isolated and correctly configured client.
  client = chromadb.PersistentClient(path=vector_db_path)

  if purge:
    if os.path.exists(vector_db_path):
      print(f"Purging contents of {vector_db_path}")
      shutil.rmtree(vector_db_path) # Remove the directory and its contents
      print(f"Purged contents of {vector_db_path}")

    print(f"Creating new vector store at {vector_db_path} after purge.")
    vector_store = Chroma.from_documents(
        documents = chunks,
        embedding = embedding_model,
        client = client, # Pass the explicit client
        persist_directory = vector_db_path
    )
    print(f"Added {len(chunks)} chunks to new vector store.")
    return vector_store

  # If not purging
  if not os.path.exists(vector_db_path):
    print(f"Creating new vector store at {vector_db_path}.")
    vector_store = Chroma.from_documents(
        documents = chunks,
        embedding = embedding_model,
        client = client, # Pass the explicit client
        persist_directory = vector_db_path
    )
    print(f"Added {len(chunks)} chunks to new vector store.")
  else:
    print(f"Loading existing vector store from {vector_db_path}.")
    vector_store = Chroma(
        client = client, # Pass the explicit client
        embedding_function=embedding_model,
        persist_directory=vector_db_path
    )

    # Get unique source file paths from the new chunks being added
    new_chunk_sources = set(doc.metadata['source'] for doc in chunks if 'source' in doc.metadata)

    if new_chunk_sources:
        print(f"Found new chunk sources to update: {new_chunk_sources}")
        for source_file in new_chunk_sources:
            # Delete existing documents from this source file in the vector store
            print(f"Deleting existing chunks in DB from source: {source_file}")
            vector_store.delete(where={"source": source_file})
        print(f"Deleted existing chunks for the specified sources.")

    print(f"Adding {len(chunks)} new chunks to existing vector store.")
    vector_store.add_documents(documents=chunks)

  return vector_store

## Method to do similarity search

In [36]:
def vector_db_similarity_search(vector_db, query, k=5):
  # results = vector_db.similarity_search(query=query, k=k)
  # Increase diversity, so use lambda_mult=0.1
  results = vector_db.max_marginal_relevance_search(query=query, k=k, lambda_mult=0.1)
  print(f"Similiarity search returned {len(results)} chunks")
  return results

## Use LLM to generate text

In [30]:
from groq import Groq
def generate_response(system_role: str, query: str):
  groq_client = Groq(api_key = groq_key)
  try:
    result = groq_client.chat.completions.create(
        model = "llama-3.1-8b-instant",
        messages = [
            {
                "role":"system",
                "content": system_role
            },
            {
                "role":"user",
                "content":query
            }
        ]
    )
    return result.choices[0].message.content
  except Exception as e:
    print(f"Groq Connection failed: {e}")

## Rag Pipeline

## This for testing individual functions

In [11]:

from langchain_huggingface import HuggingFaceEmbeddings

# Get the docs
documents = load_documents(r'/content/raw_documents')
# print(type(documents))

#chunk the docs
chunks  = chunkizer(documents= documents, chunk_size= 1000, chunk_overlap=200)
print(type(chunks))
print(chunks[0])

# create embedding model
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
print(type(embedding_model))

# add chunks to vector db
add_to_vector_db(r'/content/dbs', chunks=chunks, embedding_model=embedding_model, purge=False)



Loaded 6 pages from /content/raw_documents/AST-2.pdf
Loaded 5 pages from /content/raw_documents/AST-1.pdf
Loaded total 11 documents
Chunizer: num documents: 11, Chunk Sz: 1000 Chunk OverLap: 200
Chunkizer created 24 chunks
<class 'list'>
page_content='Module 3: AST-2 
 
TITLE: Testing the Modules & Packaging  
 
LEARNING OBJECTIVES: 
At the end of the experiment, you will be able to include testing aspects in the project and 
write test cases continuing from AST1. Finally, you will be able to create a python package 
of the model which can be easily consumed by any API. 
 
You will be able to understand and implement the following aspects: 
 
1. Testing concept and automated testing using pytest  
2. Packaging of model  
 
INTRODUCTION 
Testing: Software testing is a crucial part of the software development process. It 
involves executing a program or system with the intention of finding errors or verifying its 
compliance with specified requirements. The goal of testing is to identify

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
Loading existing vector store from /content/dbs.
Found new chunk sources to update: {'/content/raw_documents/AST-2.pdf', '/content/raw_documents/AST-1.pdf'}
Deleting existing chunks in DB from source: /content/raw_documents/AST-2.pdf
Deleting existing chunks in DB from source: /content/raw_documents/AST-1.pdf
Deleted existing chunks for the specified sources.
Adding 24 new chunks to existing vector store.


/tmp/ipykernel_12490/1988614297.py:29: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory=vector_db_path, embedding_function=embedding_model)


## Gradio Application and the actual pipeline

In [31]:
import gradio as gr
import os
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

DEFAULT_DOCS_PATH = '/content/raw_documents'
DEFAULT_DB_PATH = '/content/dbs'

# Initialize the embedding model once globally
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

def rag_pipeline(document_choice: str, uploaded_files, question: str, session_id: str):
    global embedding_model # Use the globally initialized embedding model

    current_vector_db = None

    # Define session-specific paths
    SESSION_USER_DOCS_PATH = os.path.join('/content/user_docs', session_id)
    SESSION_USER_DBS_PATH = os.path.join('/content/user_dbs', session_id)

    # Create directories if they don't exist for the current session
    os.makedirs(SESSION_USER_DOCS_PATH, exist_ok=True)
    os.makedirs(SESSION_USER_DBS_PATH, exist_ok=True)

    if document_choice == 'default':
        print(f"[{session_id}] Using default documents and vector DB.")
        if not os.path.exists(DEFAULT_DB_PATH):
            return "Default vector database not found. Please ensure it has been initialized by running the RAG pipeline cells above."
        current_vector_db = Chroma(persist_directory=DEFAULT_DB_PATH,
                                   embedding_function=embedding_model)

    elif document_choice == 'upload':
        print(f"[{session_id}] Processing uploaded documents.")
        if not uploaded_files:
            return "Please upload PDF files for processing."

        # Clear previous user documents for this session
        if os.path.exists(SESSION_USER_DOCS_PATH):
            shutil.rmtree(SESSION_USER_DOCS_PATH)
        os.makedirs(SESSION_USER_DOCS_PATH, exist_ok=True)

        # Save uploaded files to the session's document folder
        saved_file_paths = []
        for uploaded_file in uploaded_files:
            file_name = os.path.basename(uploaded_file.name)
            save_path = os.path.join(SESSION_USER_DOCS_PATH, file_name)
            shutil.copyfile(uploaded_file.name, save_path)
            saved_file_paths.append(save_path)
            print(f"[{session_id}] Saved {file_name} to {SESSION_USER_DOCS_PATH}")

        # Load, chunk, and add to the session's vector DB
        user_documents = load_documents(SESSION_USER_DOCS_PATH)
        user_chunks = chunkizer(documents=user_documents, chunk_size=1000, chunk_overlap=200)
        current_vector_db = add_to_vector_db(SESSION_USER_DBS_PATH,
                                             chunks=user_chunks,
                                             embedding_model=embedding_model,
                                             purge=True)

    else:
        return "Invalid document choice. Please select 'default' or 'upload'."

    if not current_vector_db:
        return "Error: Could not initialize vector database."

    # Perform retrieval
    print(f"[{session_id}] Performing similarity search for: {question}")
    retrieved_docs = vector_db_similarity_search(current_vector_db, query=question, k=5)

    # Combine retrieved documents into a context string
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # Generate response using LLM
    system_role = "You are a helpful assistant. Use the following context to answer the user's question. If the answer is not in the context, say 'I cannot find the answer in the provided documents.'"

    full_query = f"Context: {context}\n\nQuestion: {question}"
    print(f"[{session_id}] Generating response for query: {full_query}")
    response = generate_response(system_role, full_query)

    return response


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# RAG Document Query Interface")

    session_id = gr.State(value="") # To store the session ID

    with gr.Row():
        document_choice = gr.Radio(
            ['default', 'upload'],
            label="Choose Document Source",
            value='default'
        )

    with gr.Row() as upload_section:
        uploaded_files = gr.File(
            label="Upload PDF Documents (for 'upload' option only)",
            file_count="multiple",
            file_types=[".pdf"],
            visible=False # Initially hidden
        )
        upload_btn = gr.Button("Process Uploaded Documents", visible=False)

    def toggle_upload_visibility(choice):
        if choice == 'upload':
            return gr.update(visible=True), gr.update(visible=True)
        else:
            return gr.update(visible=False), gr.update(visible=False)

    document_choice.change(
        toggle_upload_visibility,
        inputs=document_choice,
        outputs=[uploaded_files, upload_btn]
    )

    with gr.Row():
        question_input = gr.Textbox(
            label="Find me in the documents:",
            placeholder="Enter your question here..."
        )
    with gr.Row():
        submit_btn = gr.Button("Get Answer")
    with gr.Row():
        output_text = gr.Textbox(label="Response", interactive=False, lines=10)

    def process_and_query(doc_choice, files, query_text, current_session_id):
        if doc_choice == 'upload' and not files:
            return "Please upload documents or select 'default' source."
        # Pass the session_id to rag_pipeline
        return rag_pipeline(doc_choice, files, query_text, current_session_id)

    submit_btn.click(
        process_and_query,
        inputs=[document_choice, uploaded_files, question_input, session_id],
        outputs=output_text
    )

    # Trigger processing for uploaded files if the upload button is clicked
    upload_btn.click(
        process_and_query,
        inputs=[document_choice, uploaded_files, question_input, session_id],
        outputs=output_text
    )

    # Set session_id value on load using gr.Request().session_hash
    demo.load(lambda request=None: request.session_hash if request else '', outputs=session_id)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fa5fa4c2fa754fd46b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[] Using default documents and vector DB.
[] Performing similarity search for: what is the objective of AST-1?
Similiarity search returned 5 chunks
[] Generating response for query: Context: Module 3: AST-2 
 
TITLE: Testing the Modules & Packaging  
 
LEARNING OBJECTIVES: 
At the end of the experiment, you will be able to include testing aspects in the project and 
write test cases continuing from AST1. Finally, you will be able to create a python package 
of the model which can be easily consumed by any API. 
 
You will be able to understand and implement the following aspects: 
 
1. Testing concept and automated testing using pytest  
2. Packaging of model  
 
INTRODUCTION 
Testing: Software testing is a crucial part of the software development process. It 
involves executing a program or system with the intention of finding errors or verifying its 
compliance with specified requirements. The goal of testing is to identify defects and 
ensure that the software functions as intended,

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1636, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

[] Processing uploaded documents.
[] Saved MH2_RAG_Instructions.pdf to /content/user_docs/
Loaded 1 pages from /content/user_docs/MH2_RAG_Instructions.pdf
Loaded total 1 documents
Chunizer: num documents: 1, Chunk Sz: 1000 Chunk OverLap: 200
Chunkizer created 2 chunks
Purging contents of /content/user_dbs/
Purged contents of /content/user_dbs/
Creating new vector store at /content/user_dbs/ after purge.


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1636, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^